In [1]:
pip install torch

In [4]:
"""
04_train_and_evaluate.py
──────────────────────────
STAGE 4 of 4 in the dataset pipeline.

Trains and evaluates classical ML classifiers (Logistic Regression,
Random Forest, Gradient Boosting, Linear SVM, optionally LightGBM) on
TF-IDF + hand-engineered sequence features, then generates evaluation
plots.

IMPORTANT — naming and scope, please read:
  This file was previously named train_lstm.ipynb, but contained no LSTM,
  no PyTorch model, and never imported torch despite installing it in the
  first cell. Every model actually trained was classical sklearn. I have
  renamed this file to describe what it actually does rather than silently
  add a real LSTM on your behalf, since "make the training methodology a
  sequence model" is a real methodological decision for your thesis, not
  something to change quietly during a cleanup pass.
  If you DO want an actual recurrent/sequence model (LSTM, GRU, or a small
  Transformer over the technique-ID sequences) in addition to or instead
  of these classical baselines, tell me and I'll add it as its own stage —
  it's a meaningfully different piece of code (needs a vocabulary/embedding
  layer, padding, a training loop, etc.), not a drop-in change to this file.

Other changes vs. the old script:
  - KEEP_CLASSES is now read from label_encoder.json (stage 3's output)
    instead of being hardcoded here as ["Espionage", "Financial"]. Stage 2
    is the ONLY place that decides whether Sabotage is kept (KEEP_SABOTAGE).
    Before, this file had its own independent, disconnected copy of that
    decision — if you ever flip KEEP_SABOTAGE to True in stage 2, this
    file used to still silently drop Sabotage rows because its hardcoded
    class list didn't know that. Now it can't get out of sync, because it
    isn't a separate decision anymore.
  - The sequence-length feature no longer divides by a hardcoded 70.0.
    Your dataset already contains sequences longer than that (Operation
    Wocao at 70, SolarWinds Compromise at 71), which would have silently
    pushed this feature above the intended 0-1 range. It's now computed
    from the actual max length seen in the training data.

Inputs:
  train.csv, val.csv, test.csv   <- output of 03_prepare_splits.py
  label_encoder.json             <- output of 03_prepare_splits.py

Outputs:
  results.csv
  predictions.csv
  prefix_accuracy.csv
  classification_report.txt
  plots/  (7 PNG + PDF figure pairs)
"""

import json
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    precision_recall_curve, roc_curve, auc, average_precision_score
)
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
TRAIN_FILE   = "train.csv"
VAL_FILE     = "val.csv"
TEST_FILE    = "test.csv"
LABEL_FILE   = "label_encoder.json"
RANDOM_STATE = 42
PLOTS_DIR    = "plots"

matplotlib.rcParams.update({
    "font.family":       "serif",
    "font.size":         11,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
})

C_ESP, C_FIN, C_GREY, C_VAL, C_TEST = "#2166AC", "#D6604D", "#737373", "#4DAC26", "#D01C8B"

# Semantic per-class colors, matching the original script's hardcoded
# C_ESP (blue) / C_FIN (red) scheme. Falls back to a neutral color for
# any class beyond these two (e.g. if KEEP_SABOTAGE=True is ever used).
_CLASS_COLORS = {"Espionage": C_ESP, "Financial": C_FIN}
_FALLBACK_COLORS = ["#984EA3", "#FF7F00", "#A65628"]  # used only if a 3rd+ class appears


def class_color(cls: str, classes: list) -> str:
    if cls in _CLASS_COLORS:
        return _CLASS_COLORS[cls]
    others = [c for c in classes if c not in _CLASS_COLORS]
    return _FALLBACK_COLORS[others.index(cls) % len(_FALLBACK_COLORS)]


# ── Data loading ──────────────────────────────────────────────────────────────
def load_data():
    with open(LABEL_FILE) as f:
        label_map = json.load(f)
    # Classes come from stage 3's label encoder, which reflects whatever
    # KEEP_SABOTAGE decision stage 2 made -- no separate copy of that
    # decision lives here.
    classes = sorted(label_map, key=label_map.get)
    print(f"    Classes (from {LABEL_FILE}): {classes}")

    train_df = pd.read_csv(TRAIN_FILE)
    val_df   = pd.read_csv(VAL_FILE)
    test_df  = pd.read_csv(TEST_FILE)

    train_df = train_df[train_df["motive"].isin(classes)].reset_index(drop=True)
    val_df   = val_df[val_df["motive"].isin(classes)].reset_index(drop=True)
    test_df  = test_df[test_df["motive"].isin(classes)].reset_index(drop=True)

    for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        counts = " ".join(f"{c}: {(d['motive'] == c).sum()}" for c in classes)
        print(f"    {name} actors: {len(d)}  ({counts})")

    return train_df, val_df, test_df, classes, label_map


# ── Prefix expansion ──────────────────────────────────────────────────────────
def expand_prefixes(df, label_map):
    samples = []
    for _, row in df.iterrows():
        techniques = row["technique_id_seq"].split(" -> ")
        label      = label_map[row["motive"]]
        for end in range(1, len(techniques) + 1):
            samples.append({
                "techniques":    techniques[:end],
                "motive":        row["motive"],
                "label":         label,
                "prefix_length": end,
            })
    return samples


# ── Feature engineering ────────────────────────────────────────────────────────
def make_sequence_features(samples, vocab, max_len):
    n          = len(samples)
    vocab_list = sorted(vocab)
    v_idx      = {t: i for i, t in enumerate(vocab_list)}
    V          = len(vocab_list)
    extra      = np.zeros((n, 2 + 2 * V), dtype=np.float32)
    for i, s in enumerate(samples):
        techs       = s["techniques"]
        L           = len(techs)
        extra[i, 0] = L / max_len
        extra[i, 1] = len(set(techs)) / L
        if techs[0] in v_idx:
            extra[i, 2 + v_idx[techs[0]]] = 1.0
        if techs[-1] in v_idx:
            extra[i, 2 + V + v_idx[techs[-1]]] = 1.0
    return csr_matrix(extra)


def build_features(train_samples, val_samples, test_samples):
    def texts(s): return [" ".join(x["techniques"]) for x in s]

    tfidf = TfidfVectorizer(
        analyzer="word", token_pattern=r"\S+",
        sublinear_tf=True, min_df=2,
    )
    X_tr_tfidf = tfidf.fit_transform(texts(train_samples))
    X_v_tfidf  = tfidf.transform(texts(val_samples))
    X_te_tfidf = tfidf.transform(texts(test_samples))

    vocab = set()
    for s in train_samples:
        vocab.update(s["techniques"])

    # Max length is computed from the actual training data rather than
    # a hardcoded constant, so it stays correct as your dataset grows.
    max_len = max(len(s["techniques"]) for s in train_samples)
    print(f"    Max sequence length in training data: {max_len} "
          f"(used to normalise the sequence-length feature)")

    X_tr_seq = make_sequence_features(train_samples, vocab, max_len)
    X_v_seq  = make_sequence_features(val_samples,   vocab, max_len)
    X_te_seq = make_sequence_features(test_samples,  vocab, max_len)

    return (hstack([X_tr_tfidf, X_tr_seq]),
            hstack([X_v_tfidf,  X_v_seq]),
            hstack([X_te_tfidf, X_te_seq]))


# ── Per-prefix accuracy ───────────────────────────────────────────────────────
def accuracy_by_prefix(preds, labels, prefix_lengths):
    results = defaultdict(lambda: {"correct": 0, "total": 0})
    for pred, label, plen in zip(preds, labels, prefix_lengths):
        results[plen]["total"]   += 1
        results[plen]["correct"] += int(pred == label)
    return pd.DataFrame([
        {"prefix_length": k, "accuracy": v["correct"] / v["total"],
         "correct": v["correct"], "total": v["total"]}
        for k, v in sorted(results.items())
    ])


# ── Models ────────────────────────────────────────────────────────────────────
def get_models():
    models = {
        "LogisticRegression": LogisticRegression(
            C=1.0, class_weight="balanced", max_iter=1000,
            random_state=RANDOM_STATE, solver="lbfgs",
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, class_weight="balanced", max_depth=None,
            min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=4,
            subsample=0.8, random_state=RANDOM_STATE,
        ),
        "LinearSVM": CalibratedClassifierCV(
            LinearSVC(C=0.5, class_weight="balanced", max_iter=2000,
                      random_state=RANDOM_STATE),
            cv=3,
        ),
    }
    try:
        import lightgbm as lgb
        models["LightGBM"] = lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31,
            class_weight="balanced", random_state=RANDOM_STATE,
            verbosity=-1, n_jobs=-1,
        )
        print("    LightGBM available -- added to comparison.")
    except ImportError:
        print("    LightGBM not installed -- skipping.")
    return models


# ── Plotting ──────────────────────────────────────────────────────────────────
def save_fig(fig, name):
    os.makedirs(PLOTS_DIR, exist_ok=True)
    fig.savefig(os.path.join(PLOTS_DIR, f"{name}.png"))
    fig.savefig(os.path.join(PLOTS_DIR, f"{name}.pdf"))
    plt.close(fig)
    print(f"    Saved: {PLOTS_DIR}/{name}.png  +  .pdf")

def majority_baseline_by_prefix(pred_df: pd.DataFrame, majority_class: str) -> pd.DataFrame:
    """
    Exact majority-class baseline at each prefix length, computed directly
    from the test predictions already produced by main() -- no simulation
    needed. This is NOT a constant: it is recomputed at every prefix length
    over exactly the test rows still present there, so it correctly reflects
    that longer-sequence actors dominate as shorter ones drop out (Sec 4.9 /
    Fig 5.5's declining sample count).
    """
    return (
        pred_df.assign(is_majority=(pred_df["true_motive"] == majority_class))
               .groupby("prefix_length")["is_majority"]
               .mean()
               .rename("baseline")
               .reset_index()
    )


def generate_plots(results_df, pred_df, prefix_df, classes, label_map, best_model, majority_class):
    print("\n[*] Generating plots...")
    y_true = pred_df["true_motive"].map(label_map).values
    y_pred = pred_df["predicted_motive"].map(label_map).values

    baseline_df = majority_baseline_by_prefix(pred_df, majority_class)
    prefix_df = prefix_df.merge(baseline_df, on="prefix_length", how="left")

    is_binary = len(classes) == 2
    pos_class = classes[1] if is_binary else None
    y_prob_pos = pred_df[f"prob_{pos_class}"].values if is_binary else None

    # Plot 1: model comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Model comparison", y=1.02)
    models_list = results_df["model"].tolist()
    x, w = np.arange(len(models_list)), 0.35

    ax = axes[0]
    ax.bar(x - w/2, results_df["val_acc"],  w, label="Val accuracy",  color=C_VAL,  alpha=0.85)
    ax.bar(x + w/2, results_df["test_acc"], w, label="Test accuracy", color=C_TEST, alpha=0.85)
    ax.axhline(1 / len(classes), color=C_GREY, linestyle=":", linewidth=1, label="Random baseline")
    ax.set_xticks(x); ax.set_xticklabels(models_list, rotation=15, ha="right")
    ax.set_ylabel("Accuracy"); ax.set_ylim(0.4, 1.0)
    ax.set_title("Validation vs test accuracy"); ax.legend()
    for i, (v, t) in enumerate(zip(results_df["val_acc"], results_df["test_acc"])):
        ax.text(i - w/2, v + 0.005, f"{v:.2f}", ha="center", fontsize=8)
        ax.text(i + w/2, t + 0.005, f"{t:.2f}", ha="center", fontsize=8)

    ax = axes[1]
    for i, cls in enumerate(classes):
        col = f"f1_{cls.lower()[:3]}"
        if col in results_df.columns:
            bars = ax.bar(x + (i - (len(classes)-1)/2) * w, results_df[col], w,
                          label=f"{cls} F1", color=class_color(cls, classes), alpha=0.85)
            for xi, val in zip(x + (i - (len(classes)-1)/2) * w, results_df[col]):
                ax.text(xi, val + 0.005, f"{val:.2f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(models_list, rotation=15, ha="right")
    ax.set_ylabel("F1 score"); ax.set_ylim(0.0, 1.0)
    ax.set_title("Per-class F1 scores"); ax.legend()
    plt.tight_layout()
    save_fig(fig, "01_model_comparison")

    # Plot 2: confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=range(len(classes)))
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    fig.suptitle(f"Confusion matrix -- {best_model}", y=1.02)
    for ax, data, fmt, title in zip(axes, [cm, cm_pct], [".0f", ".2%"],
                                     ["Counts", "Row-normalised (recall per class)"]):
        im = ax.imshow(data, cmap="Blues", aspect="auto", vmin=0, vmax=data.max())
        ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
        ax.set_xticklabels(classes); ax.set_yticklabels(classes)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
        for i in range(len(classes)):
            for j in range(len(classes)):
                val = data[i, j]
                colour = "white" if val > data.max() * 0.6 else "black"
                ax.text(j, i, f"{val:{fmt}}", ha="center", va="center",
                        fontsize=13, fontweight="bold", color=colour)
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    save_fig(fig, "02_confusion_matrix")

    # Plot 3: prefix accuracy
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"Early prediction accuracy by prefix length -- {best_model}", y=1.02)
    ax1 = axes[0]; ax2 = ax1.twinx()
    ax1.plot(prefix_df["prefix_length"], prefix_df["accuracy"], color=C_TEST,
             linewidth=2, marker="o", markersize=3, label="Accuracy")
    ax1.plot(prefix_df["prefix_length"], prefix_df["baseline"], color=C_GREY,
             linestyle=":", linewidth=1.3, label=f"Majority-class baseline ({majority_class})")
    ax2.bar(prefix_df["prefix_length"], prefix_df["total"], color=C_GREY, alpha=0.15, label="Sample count")
    ax1.set_xlabel("Prefix length (number of techniques observed)")
    ax1.set_ylabel("Accuracy", color=C_TEST)
    ax2.set_ylabel("Number of test samples", color=C_GREY)
    ax1.set_ylim(0, 1.05); ax1.set_title("Accuracy over all prefix lengths")
    l1, la1 = ax1.get_legend_handles_labels(); l2, la2 = ax2.get_legend_handles_labels()
    ax1.legend(l1 + l2, la1 + la2, loc="lower right")

    ax = axes[1]
    short = prefix_df[prefix_df["prefix_length"] <= 20]
    ax.plot(short["prefix_length"], short["accuracy"], color=C_TEST, linewidth=2, marker="o", markersize=5)
    ax.fill_between(short["prefix_length"], short["baseline"], short["accuracy"],
                    where=short["accuracy"] >= short["baseline"],
                    alpha=0.15, color=C_TEST, label="Above baseline")
    ax.fill_between(short["prefix_length"], short["baseline"], short["accuracy"],
                    where=short["accuracy"] < short["baseline"],
                    alpha=0.30, color=C_GREY, label="Below baseline")
    ax.plot(short["prefix_length"], short["baseline"], color=C_GREY,
            linestyle=":", linewidth=1.3, label=f"Majority-class baseline ({majority_class})")
    ax.set_xlabel("Prefix length (number of techniques observed)")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0.3, 1.05); ax.set_xticks(range(1, 21))
    ax.set_title("Early prediction -- first 20 techniques"); ax.legend()
    for _, row in short.iterrows():
        ax.annotate(f"{row['accuracy']:.0%}",
                    (row["prefix_length"], row["accuracy"]),
                    textcoords="offset points", xytext=(0, 6),
                    ha="center", fontsize=7, color=C_TEST)
    plt.tight_layout()
    save_fig(fig, "03_prefix_accuracy")

    if is_binary:
        # Plot 4: precision-recall (binary only)
        prec, rec, thresh = precision_recall_curve(y_true, y_prob_pos)
        ap = average_precision_score(y_true, y_prob_pos)
        baseline = y_true.mean()
        fig, ax = plt.subplots(figsize=(7, 5.5))
        ax.plot(rec, prec, color=C_FIN, linewidth=2, label=f"{pos_class} (AP = {ap:.3f})")
        ax.axhline(baseline, color=C_GREY, linestyle=":", linewidth=1, label=f"Random baseline ({baseline:.2f})")
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
        ax.set_title(f"Precision-Recall curve -- {best_model}\n(positive class: {pos_class})")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.05); ax.legend()
        idx = np.argmin(np.abs(thresh - 0.5))
        ax.scatter(rec[idx], prec[idx], color=C_FIN, s=80, zorder=5,
                  label=f"t=0.5  P={prec[idx]:.2f}  R={rec[idx]:.2f}")
        ax.annotate(f"  t=0.5\n  P={prec[idx]:.2f}, R={rec[idx]:.2f}", (rec[idx], prec[idx]), fontsize=9)
        ax.legend()
        save_fig(fig, "04_precision_recall")

        # Plot 5: ROC (binary only)
        fpr, tpr, _ = roc_curve(y_true, y_prob_pos)
        roc_auc = auc(fpr, tpr)
        fig, ax = plt.subplots(figsize=(6.5, 5.5))
        ax.plot(fpr, tpr, color=C_FIN, linewidth=2, label=f"{pos_class} (AUC = {roc_auc:.3f})")
        ax.plot([0, 1], [0, 1], color=C_GREY, linestyle=":", linewidth=1, label="Random (AUC = 0.500)")
        ax.fill_between(fpr, tpr, alpha=0.08, color=C_FIN)
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        ax.set_title(f"ROC curve -- {best_model}")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.legend(loc="lower right")
        save_fig(fig, "05_roc_curve")
    else:
        print("    Skipping precision-recall / ROC plots (only defined for binary classification)")

    # Plot 6: confidence distribution
    correct_mask = (y_true == y_pred)
    confidence_all = pred_df["confidence"].values
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"Prediction confidence distribution -- {best_model}", y=1.02)
    ax = axes[0]
    for i, cls in enumerate(classes):
        mask = y_true == i
        ax.hist(confidence_all[mask], bins=20, range=(1/len(classes), 1.0),
                alpha=0.6, color=class_color(cls, classes), label=cls,
                edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Prediction confidence"); ax.set_ylabel("Number of prefix samples")
    ax.set_title("Confidence by true class"); ax.legend()
    ax = axes[1]
    ax.hist(confidence_all[correct_mask], bins=20, range=(1/len(classes), 1.0),
            alpha=0.6, color=C_VAL, label="Correct", edgecolor="white", linewidth=0.5)
    ax.hist(confidence_all[~correct_mask], bins=20, range=(1/len(classes), 1.0),
            alpha=0.6, color=C_FIN, label="Incorrect", edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Prediction confidence"); ax.set_ylabel("Number of prefix samples")
    ax.set_title("Confidence: correct vs incorrect predictions"); ax.legend()
    plt.tight_layout()
    save_fig(fig, "06_confidence_dist")

    # Plot 7: metrics heatmap across models
    f1_cols = [c for c in results_df.columns if c.startswith("f1_")] + ["macro_f1", "val_acc", "test_acc"]
    metrics = results_df[["model"] + f1_cols].set_index("model")
    # Pretty labels: "f1_esp" -> "Espionage F1", etc., falling back to the
    # raw column name for anything not recognized (e.g. a class whose
    # 3-letter code doesn't match a known name).
    pretty_names = {f"f1_{c.lower()[:3]}": f"{c} F1" for c in classes}
    pretty_names.update({"macro_f1": "Macro F1", "val_acc": "Val Acc", "test_acc": "Test Acc"})
    metrics.columns = [pretty_names.get(c, c) for c in metrics.columns]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    im = ax.imshow(metrics.values.T, cmap="RdYlGn", aspect="auto", vmin=0.4, vmax=1.0)
    ax.set_xticks(range(len(metrics.index))); ax.set_yticks(range(len(metrics.columns)))
    ax.set_xticklabels(metrics.index, rotation=15, ha="right")
    ax.set_yticklabels(metrics.columns)
    ax.set_title("Performance metrics heatmap -- all models")
    for i in range(len(metrics.columns)):
        for j in range(len(metrics.index)):
            val = metrics.values[j, i]
            colour = "black" if 0.45 < val < 0.85 else "white"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=10, color=colour)
    plt.colorbar(im, ax=ax, fraction=0.03, label="Score")
    plt.tight_layout()
    save_fig(fig, "07_metrics_heatmap")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print("[*] Loading data...")
    train_df, val_df, test_df, classes, label_map = load_data()

    print("\n[*] Expanding prefixes...")
    train_samples = expand_prefixes(train_df, label_map)
    val_samples   = expand_prefixes(val_df,   label_map)
    test_samples  = expand_prefixes(test_df,  label_map)
    print(f"    Train samples: {len(train_samples)}")
    print(f"    Val samples:   {len(val_samples)}")
    print(f"    Test samples:  {len(test_samples)}")

    print("\n[*] Building features (TF-IDF + sequence features)...")
    X_train, X_val, X_test = build_features(train_samples, val_samples, test_samples)

    y_train = np.array([s["label"] for s in train_samples])
    y_val   = np.array([s["label"] for s in val_samples])
    y_test  = np.array([s["label"] for s in test_samples])
    prefix_lengths_test = np.array([s["prefix_length"] for s in test_samples])
    true_motives_test   = [s["motive"] for s in test_samples]

    print(f"    Feature matrix: {X_train.shape[1]} features\n")

    models = get_models()
    results, best_val_acc, best_name, best_preds, best_probs, best_test_acc = [], -1, "", None, None, -1

    header = f"{'Model':<22} {'Val Acc':>8} {'Test Acc':>9} " + \
             " ".join(f"{c[:3]} F1".rjust(9) for c in classes)
    print(header)
    print("-" * len(header))

    for name, clf in models.items():
        clf.fit(X_train, y_train)
        val_preds  = clf.predict(X_val)
        test_preds = clf.predict(X_test)
        test_probs = clf.predict_proba(X_test)

        val_acc  = accuracy_score(y_val,  val_preds)
        test_acc = accuracy_score(y_test, test_preds)
        rd = classification_report(y_test, test_preds, target_names=classes,
                                    output_dict=True, zero_division=0)

        row = {"model": name, "val_acc": round(val_acc, 4), "test_acc": round(test_acc, 4),
               "macro_f1": round(rd["macro avg"]["f1-score"], 4)}
        f1_str = ""
        for c in classes:
            f1 = rd.get(c, {}).get("f1-score", 0)
            row[f"f1_{c.lower()[:3]}"] = round(f1, 4)
            f1_str += f"{f1:>9.4f}"
        print(f"{name:<22} {val_acc:>8.4f} {test_acc:>9.4f} {f1_str}")

        results.append(row)
        if val_acc > best_val_acc:
            best_val_acc, best_name = val_acc, name
            best_preds, best_probs, best_test_acc = test_preds, test_probs, test_acc

    results_df = pd.DataFrame(results).sort_values("test_acc", ascending=False)
    results_df.to_csv("results.csv", index=False)
    print(f"\n[OK] Best model (selected on validation accuracy): {best_name} "
          f"(val acc: {best_val_acc:.4f}, test acc: {best_test_acc:.4f})")

    report_str = classification_report(y_test, best_preds, target_names=classes,
                                        digits=4, zero_division=0)
    print(f"\n[*] Classification report ({best_name}):\n")
    print(report_str)
    with open("classification_report.txt", "w") as f:
        f.write(f"Best model: {best_name}\n\n{report_str}")

    pred_df = pd.DataFrame({
        "prefix_length":    prefix_lengths_test,
        "true_motive":      true_motives_test,
        "predicted_motive": [classes[p] for p in best_preds],
        "confidence":       [max(p) for p in best_probs],
    })
    for i, cname in enumerate(classes):
        pred_df[f"prob_{cname}"] = [p[i] for p in best_probs]
    pred_df.to_csv("predictions.csv", index=False)

    prefix_df = accuracy_by_prefix(best_preds, y_test, prefix_lengths_test)
    prefix_df.to_csv("prefix_accuracy.csv", index=False)

    majority_class = train_df["motive"].value_counts().idxmax()

    generate_plots(results_df, pred_df, prefix_df, classes, label_map, best_name, majority_class)  # <- add majority_class

    print("\n[*] Accuracy by prefix length (best model):")
    print(f"{'Prefix':>8} {'Accuracy':>10} {'Samples':>9}")
    print("-" * 30)
    for _, row in prefix_df.iterrows():
        print(f"{int(row['prefix_length']):>8} {row['accuracy']:>10.4f} {int(row['total']):>9}")

    print("\n[OK] Outputs saved:")
    print("    results.csv")
    print("    predictions.csv")
    print("    prefix_accuracy.csv")
    print("    classification_report.txt")
    print("    plots/  (7 figures, PNG + PDF)")


if __name__ == "__main__":
    main()

[*] Loading data...
    Classes (from label_encoder.json): ['Espionage', 'Financial']
    Train actors: 131  (Espionage: 78 Financial: 53)
    Val actors: 28  (Espionage: 17 Financial: 11)
    Test actors: 29  (Espionage: 17 Financial: 12)

[*] Expanding prefixes...
    Train samples: 2949
    Val samples:   693
    Test samples:  626

[*] Building features (TF-IDF + sequence features)...
    Max sequence length in training data: 103 (used to normalise the sequence-length feature)
    Feature matrix: 554 features

    LightGBM available -- added to comparison.
Model                   Val Acc  Test Acc    Esp F1    Fin F1
-------------------------------------------------------------
LogisticRegression       0.7965    0.8546    0.8841   0.8051
RandomForest             0.7706    0.7987    0.8471   0.7056
GradientBoosting         0.7864    0.7827    0.8440   0.6421
LinearSVM                0.7922    0.7732    0.8356   0.6340
LightGBM                 0.7994    0.8371    0.8747   0.7671

[OK